# Lesson 5 — Train/Test Split and First Classifier / Podział na train/test i pierwszy klasyfikator

Read `README.md` (or `README.pl.md`) first — it has Meridian Outlet's framing for today. You built a baseline that never caught a return; now fit a real model and see if it does any better. One rule for this whole notebook: `test_df` gets created below and then set aside — nothing in this lesson reads its contents in any way, not even to count customers. It's opened for the first time in Lesson 6, after the threshold is chosen.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from task import load_and_merge_orders

df = load_and_merge_orders()
df.shape

## Split before you look / Podziel, zanim zajrzysz

With only 14% of orders returned, a careless split could give the test set a very different return rate than the train set.

In [ ]:
from task import split_orders

train_df, test_df = split_orders(df)
len(train_df), len(test_df), train_df["is_returned"].mean(), test_df["is_returned"].mean()

## Wait — does a test set like this contain "new" customers? / Czy taki zbiór testowy zawiera "nowych" klientów?

Not necessarily — this is a row-level split, and it doesn't know or care whether the same customer shows up on both sides. We want to see how much that matters, but without reading anything out of the actual `test_df` above — even a harmless-looking count. So carve a demonstration `fit`/`val` pair out of `train_df` instead, the same way Lesson 6 will for real, and check the overlap there.

In [ ]:
row_fit_df, row_val_df = split_orders(train_df)

row_overlap = set(row_fit_df["customer_id"]) & set(row_val_df["customer_id"])
pct_val_rows_from_known_customers = row_val_df["customer_id"].isin(row_overlap).mean()
len(row_overlap), pct_val_rows_from_known_customers

## Fit your first real classifier / Dopasuj swój pierwszy prawdziwy klasyfikator

Using the three signals you found in Lesson 3: discount percent, return history, and account age.

In [ ]:
from task import fit_classifier

model = fit_classifier(train_df)
model.coef_

## Predict at the default threshold — in-sample, not on `test_df` / Predykcja przy domyślnym progu — w próbie, nie na `test_df`

We want to see what threshold 0.5 does to this model's catch rate. We could check that on `test_df` — but that would mean looking at test-set results before Lesson 6 has even chosen a threshold, which is exactly the habit this case wants you to avoid. So check it in-sample instead: predict on `train_df`, the exact data the model was fit on. If the model can't catch a return even on data it's already seen, that's a threshold problem you don't need held-out data to see. Compare `tp` here to Lesson 4's baseline.

In [ ]:
from sklearn.metrics import confusion_matrix
from task import predict_return

predicted = predict_return(model, train_df)
confusion_matrix(train_df["is_returned"], predicted, labels=[0, 1])

## But look closer — the label alone doesn't tell the whole story / Ale spójrz bliżej — sama etykieta nie mówi wszystkiego

`predict_return` collapses every prediction to a hard 0 or 1 at the 0.5 threshold. Before you conclude the model learned nothing, look at the actual predicted probabilities (still in-sample).

In [ ]:
from task import FEATURE_COLUMNS

proba = model.predict_proba(train_df[FEATURE_COLUMNS])[:, 1]
proba.min(), proba.max(), proba.mean()

## Row split vs. group split — a validation-level comparison / Podział wierszowy vs. grupowy — porównanie na walidacji

The demonstration row-level split above (`row_fit_df`/`row_val_df`) left most customers on both sides. To see what a group-level split would change, split `train_df` by customer instead, and compare both on their own `fit`/`val` pair — the same kind of validation holdout Lesson 6 will use for real. This keeps the comparison honest: it never touches `test_df`. We'll use ROC-AUC alongside the confusion matrix — a threshold-free way to compare ranking quality (0.5 is random, 1.0 is perfect) — since both confusion matrices below are degenerate at the 0.5 threshold.

In [ ]:
from task import split_orders_by_customer

grp_fit_df, grp_val_df = split_orders_by_customer(train_df)

grp_overlap_val = set(grp_fit_df["customer_id"]) & set(grp_val_df["customer_id"])
row_sizes = (len(row_fit_df), len(row_val_df), len(row_overlap))
grp_sizes = (len(grp_fit_df), len(grp_val_df), len(grp_overlap_val))
row_sizes, grp_sizes

In [ ]:
from sklearn.metrics import roc_auc_score

row_model = fit_classifier(row_fit_df)
row_pred = predict_return(row_model, row_val_df)
row_proba = row_model.predict_proba(row_val_df[FEATURE_COLUMNS])[:, 1]

grp_model = fit_classifier(grp_fit_df)
grp_pred = predict_return(grp_model, grp_val_df)
grp_proba = grp_model.predict_proba(grp_val_df[FEATURE_COLUMNS])[:, 1]

row_cm = confusion_matrix(row_val_df["is_returned"], row_pred, labels=[0, 1])
grp_cm = confusion_matrix(grp_val_df["is_returned"], grp_pred, labels=[0, 1])
row_auc = roc_auc_score(row_val_df["is_returned"], row_proba)
grp_auc = roc_auc_score(grp_val_df["is_returned"], grp_proba)

print("row split  (fit->val): cm=", row_cm.tolist(), " auc=", row_auc)
print("group split (fit->val): cm=", grp_cm.tolist(), " auc=", grp_auc)

## Your notes / Twoje notatki

Write down: given the in-sample probability range you just saw, is this model actually useless, or is 0.5 simply the wrong threshold for Meridian Outlet's problem? What do you think would happen to the confusion matrix if you lowered the threshold?

Also: why are the row-split and group-split numbers above close rather than dramatically different? What property of `previous_returns_count`/`account_age_days` (check `data/generate.py`) makes that less surprising than it might first seem?

One more thing to note before you close this notebook: `test_df` was created above and never read again anywhere in this lesson — not to count customers, not for the threshold-0.5 check, not for the split comparison. Lesson 6 opens it exactly once, after the threshold is already chosen.